# Critical Input DEQN: Bottleneck-Adjusted Taylor Rule

This notebook trains the bottleneck-adjusted Taylor-rule DEQN network using the frozen natural benchmark checkpoint.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
NATURAL_CKPT = ARTIFACT_ROOT / 'natural' / 'natural.pt'
OUT = ARTIFACT_ROOT / 'modified_taylor'
OUT.mkdir(parents=True, exist_ok=True)

RULE_STEPS = 50_000
QMC_TRAIN = 512
QMC_VAL = 4096
N_VAL_STATES = 4096
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
TARGET_RMS = None
TARGET_MAX_ABS = None
EARLY_STOP_PATIENCE = None
MIN_STEPS_BEFORE_STOP = None
STOP_VAL_STATES = 2048
DEVICE = 'cpu'
DTYPE = 'float64'

if not NATURAL_CKPT.exists():
    raise FileNotFoundError(f'Missing natural benchmark checkpoint: {NATURAL_CKPT}')
print(ROOT)
print(OUT)

In [ ]:
cmd = [
    sys.executable, '-m', 'src.critical_input_deqn.run_train',
    '--output-dir', str(OUT),
    '--policies', 'ba',
    '--natural-checkpoint', str(NATURAL_CKPT),
    '--rule-steps', str(RULE_STEPS),
    '--rule-trainer', 'episode',
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
]
if TARGET_RMS is not None:
    cmd += ['--target-rms', str(TARGET_RMS)]
if TARGET_MAX_ABS is not None:
    cmd += ['--target-max-abs', str(TARGET_MAX_ABS)]
if EARLY_STOP_PATIENCE is not None:
    cmd += ['--early-stop-patience', str(EARLY_STOP_PATIENCE)]
if MIN_STEPS_BEFORE_STOP is not None:
    cmd += ['--min-steps-before-stop', str(MIN_STEPS_BEFORE_STOP)]
subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
with (OUT / 'ba_eval.json').open('r', encoding='utf-8') as fh:
    ba_eval = json.load(fh)
ba_eval